# 01 Fetch & Chunk

AAPL/MSFT/GOOGL の 10-K × 5年 + 10-Q × 15四半期 = 60 件を取得し、
Item 1A (Risk Factors) と Item 7/Item 2 (MD&A) を抽出、
FinBERT トークナイザで 510 トークンチャンクに分割する。

In [ ]:
# Cell 1: imports + setup (必ず最初に _helpers を import)
# Jupyter のモジュールキャッシュ対策: 古い _helpers / edgar を退避してから再 import
import sys
from pathlib import Path
for _m in [m for m in list(sys.modules) if m == '_helpers' or m == 'edgar' or m.startswith('edgar.')]:
    del sys.modules[_m]
sys.path.insert(0, str(Path.cwd()))
import _helpers
_ = _helpers.setup_edgar()
device = _helpers.get_device()
print('device:', device)
print('DATA_DIR:', _helpers.DATA_DIR)


In [ ]:
# Cell 2: 10-K を 3 銘柄 × 5 年取得 (edgartools 直接利用)
import edgar
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

TICKERS = ['AAPL', 'MSFT', 'GOOGL']

def _fetch(ticker, form, limit):
    try:
        return ticker, list(edgar.Company(ticker).get_filings(form=form).head(limit))
    except Exception as e:  # noqa: BLE001
        return ticker, e

with ThreadPoolExecutor(max_workers=3) as ex:
    filings_10k = dict(ex.map(lambda t: _fetch(t, '10-K', 5), TICKERS))
for t, r in filings_10k.items():
    print(t, len(r) if not isinstance(r, Exception) else f'ERR: {r}')


In [ ]:
# Cell 3: 10-Q を 3 銘柄 × 15 四半期取得
with ThreadPoolExecutor(max_workers=3) as ex:
    filings_10q = dict(ex.map(lambda t: _fetch(t, '10-Q', 15), TICKERS))
for t, r in filings_10q.items():
    print(t, len(r) if not isinstance(r, Exception) else f'ERR: {r}')


In [ ]:
# Cell 4: filings メタを 1 つの DataFrame に統合し filings.parquet 保存
import pandas as pd

all_filing_objs = []
for d, label in [(filings_10k, '10-K'), (filings_10q, '10-Q')]:
    for ticker, result in d.items():
        if isinstance(result, Exception):
            continue
        for f in result:
            all_filing_objs.append((ticker, label, f))

df_filings = pd.DataFrame([
    {
        'filing_id': str(f.accession_number),
        'ticker': ticker,
        'form': form,
        'filing_date': pd.Timestamp(str(f.filing_date)),
        'accession_number': str(f.accession_number),
    }
    for ticker, form, f in all_filing_objs
])
df_filings = df_filings.sort_values(['ticker', 'form', 'filing_date']).reset_index(drop=True)
df_filings.to_parquet(_helpers.FILINGS_PARQUET)
print('saved:', _helpers.FILINGS_PARQUET, 'rows:', len(df_filings))
df_filings.head()


In [ ]:
# Cell 5: 10-K のセクション抽出 (Item 1A, Item 7)
section_rows = []
miss = []
for ticker, form, f in tqdm([x for x in all_filing_objs if x[1] == '10-K'], desc='10-K text'):
    try:
        text = f.text()
    except Exception as e:  # noqa: BLE001
        print(f'10-K text fail: {ticker} {f.accession_number} {e}')
        continue
    fid = str(f.accession_number)
    secs = _helpers.extract_sections(text, _helpers.SECTION_PATTERNS_10K)
    for key in ['item_1a', 'item_7']:
        if key in secs:
            section_rows.append({
                'filing_id': fid, 'section_key': key,
                'text': secs[key], 'char_count': len(secs[key]),
            })
        else:
            miss.append((ticker, fid, '10-K', key))
print('10-K sections extracted:', len(section_rows), 'miss:', len(miss))


In [ ]:
# Cell 6: 10-Q のセクション抽出 (SECTION_PATTERNS_10Q)
for ticker, form, f in tqdm([x for x in all_filing_objs if x[1] == '10-Q'], desc='10-Q text'):
    try:
        text = f.text()
    except Exception as e:  # noqa: BLE001
        print(f'10-Q text fail: {ticker} {f.accession_number} {e}')
        continue
    fid = str(f.accession_number)
    secs = _helpers.extract_sections(text, _helpers.SECTION_PATTERNS_10Q)
    for key in ['item_1a', 'item_7']:
        if key in secs:
            section_rows.append({
                'filing_id': fid, 'section_key': key,
                'text': secs[key], 'char_count': len(secs[key]),
            })
        else:
            miss.append((ticker, fid, '10-Q', key))
print('total sections:', len(section_rows), 'total miss:', len(miss))


In [ ]:
# Cell 7: sections.parquet 保存
df_sections = pd.DataFrame(section_rows)
df_sections.to_parquet(_helpers.SECTIONS_PARQUET)
print('saved:', _helpers.SECTIONS_PARQUET, 'rows:', len(df_sections))
df_sections.groupby('section_key').size()


In [ ]:
# Cell 8: FinBERT トークナイザでチャンク化
from tqdm.auto import tqdm
tokenizer, _model = _helpers.load_finbert()
del _model  # チャンク化はトークナイザのみ必要

chunk_rows = []
for row in tqdm(df_sections.to_dict('records'), desc='chunking'):
    chunks = _helpers.chunk_text(row['text'], tokenizer)
    for i, c in enumerate(chunks):
        chunk_rows.append({
            'filing_id': row['filing_id'],
            'section_key': row['section_key'],
            'chunk_idx': i,
            'text': c,
            'token_count': len(tokenizer.encode(c, add_special_tokens=False)),
        })
print('total chunks:', len(chunk_rows))


In [ ]:
# Cell 9: chunks.parquet 保存 + ticker/form 結合
df_chunks = pd.DataFrame(chunk_rows).merge(
    df_filings[['filing_id', 'ticker', 'form', 'filing_date']],
    on='filing_id', how='left',
)
df_chunks.to_parquet(_helpers.CHUNKS_PARQUET)
print('saved:', _helpers.CHUNKS_PARQUET, 'rows:', len(df_chunks))
df_chunks.groupby(['ticker', 'form', 'section_key']).size()
